# Two grasshoppers fit a line

Three sunrise photos, the hour each was taken, and a line that predicts the hour from how high the sun stands: $\hat{y} = w\,z_2 + b$. On the slides, two grasshoppers learn the two numbers of that line, one grasshopper for $w$ and one for $b$. Each hops a little, $\varepsilon$, to measure the slope of the average loss under its feet, and then jumps against that slope. They go on until the ground is flat under both of them.

This notebook is that gradient descent, in a few lines of Python. The page of the slides does the same computation one click at a time, with the same numbers, so you can check yours against it:

https://eth-bmai-hs26.github.io/BMAI-PAGE/viz/we4/grasshopper-regression/

**Your part: three tasks, each marked with 🎯, ten short lines in all.** The average loss in part 2, the gradient, that is the two slopes, in part 3, and the loop of the two grasshoppers in part 4. Everything else is done for you. After each task, run the cell marked ✅ below it: it tries your lines and prints PASS for every check they pass, or NOT YET with your numbers and the expected ones. At the end, part 5 says in large letters whether your grasshoppers found the line. The cells that show only a title draw the pictures or run the checks: run them as they are, there is nothing to change inside. Their Show code button opens one if you are curious. Every task has a hint below it, folded until you click it.

In [ ]:
#@title Imports, and the checks of the ✅ cells { display-mode: "form" }
import math

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display


def shown(value):
    # A number to four decimals, a pair as two numbers, anything else as Python writes it.
    if isinstance(value, (tuple, list)):
        return "(" + ", ".join(shown(v) for v in value) + ")"
    if isinstance(value, int) and not isinstance(value, bool):
        return str(value)
    if isinstance(value, float):
        return f"{value:.3e}" if not math.isfinite(value) or abs(value) >= 1e6 else f"{value:.4f}"
    return repr(value)


def close(mine, goal):
    # True when mine is a number within 0.00005 of goal: the same to four decimals.
    return isinstance(mine, (int, float)) and math.isfinite(mine) and abs(mine - goal) < 0.00005


def check(what, yours, expected):
    # PASS when yours is the expected number, or the expected numbers, to four decimals;
    # NOT YET otherwise, with both.
    mine = list(yours) if isinstance(yours, (tuple, list)) else [yours]
    goal = list(expected) if isinstance(expected, (tuple, list)) else [expected]
    same = len(mine) == len(goal) and all(close(m, g) for m, g in zip(mine, goal))
    if same:
        print(f"PASS     {what}")
    else:
        print(f"NOT YET  {what}")
        print(f"         yours:    {shown(yours)}")
        print(f"         expected: {shown(expected)}")


def check_that(what, holds, otherwise):
    # PASS when it holds, NOT YET with the reason otherwise.
    if holds:
        print(f"PASS     {what}")
    else:
        print(f"NOT YET  {what}")
        print(f"         {otherwise}")

## 1. Three photos

In every photo the sun stands somewhere on a grid of $4 \times 4$ squares, $z_1$ squares from the left and $z_2$ squares up. Where it stands across tells the season, from spring at the left, $z_1 = 1$, through summer, $z_1 = 2$, to autumn at the right, $z_1 = 3$. Every photo also comes with the hour it was taken, $y$, and the hour is how high the sun stands: the line reads $z_2$ only, as on the slides, so the list below keeps $z_2$ and $y$.

In [ ]:
#@title The three photos { display-mode: "form" }
# The photos of the slides: the sun at (z1, z2) on a 4 x 4 grid, and the hour y.
# Where the sun stands across, z1, tells the season: spring at 1, summer at 2, autumn at 3.
suns = [(1, 3, 9), (3, 1, 7), (2, 2, 8)]
seasons = {1: "spring", 2: "summer", 3: "autumn"}

fig, axes = plt.subplots(1, 3, figsize=(10, 4.2))
for number, (ax, (z1, z2, y)) in enumerate(zip(axes, suns), start=1):
    for k in (1, 2, 3):                                   # the grid
        ax.plot([k, k], [0, 4], color="#C9C6C0", linewidth=1)
        ax.plot([0, 4], [k, k], color="#C9C6C0", linewidth=1)
    ax.add_patch(plt.Circle((z1, z2), 0.4, facecolor="#F0C24A", edgecolor="#B07800", linewidth=1.5))
    ax.set(xlim=(0, 4), ylim=(0, 4), aspect="equal", xticks=[0, 1, 2, 3, 4], yticks=[0, 1, 2, 3, 4])
    ax.set_xticklabels(["0", "1\nspring", "2\nsummer", "3\nautumn", "4"])
    ax.set_title(f"photo {number}: {seasons[z1]}, {y} o'clock", fontsize=11)
    ax.set_xlabel(f"$z_1 = {z1}$, $z_2 = {z2}$")
plt.tight_layout()
plt.show()

In [ ]:
# The three photos, as (z2, y): how high the sun stands, and the hour each was taken.
photos = [(3, 9),    # photo 1, spring, 9 o'clock
          (1, 7),    # photo 2, autumn, 7 o'clock
          (2, 8)]    # photo 3, summer, 8 o'clock

## 2. How wrong is a line?

A line is two numbers, $w$ and $b$. For a photo with its sun at the height $z_2$, the line predicts the hour $\hat{y} = w\,z_2 + b$. Its loss on that photo is the squared miss, $\mathscr{L}_i = (y_i - \hat{y}_i)^2$, and the **average loss** $\bar{\mathscr{L}}$ is the average of the three losses. The function `average_loss(w, b)` computes it for any line.

At $w = 0$ and $b = 0$ every prediction is $0$, so the three losses are $81$, $49$ and $64$, and their average is $64.6667$, the first average on the page. The cell prints it.

🎯 **Task 1.** Complete the three lines of `average_loss`:

* `y_hat`, the hour the line predicts for the photo,
* `total` plus the loss of the photo, its squared miss,
* the average loss, the total divided by the number of photos.

Then run the cell, and the ✅ cell below it.

<details>
<summary><b>Hint for task 1</b> (click to open)</summary>

The prediction is the formula of the line, $w\,z_2 + b$, with `*` for the product; the photo's $z_2$ is `z2`. The squared miss $(y - \hat{y})^2$ uses `**`, which raises to a power, and `total = total + ...` adds it to the total. The number of photos is `len(photos)`.

</details>

In [ ]:
# ============================================================
# 🎯 TASK 1: the average loss
# ============================================================
def average_loss(w, b):
    # The average loss of the line y_hat = w * z2 + b on the three photos.
    total = 0
    for z2, y in photos:
        # 🎯 COMPLETE THIS LINE: the hour the line predicts for this photo
        y_hat = ____
        # 🎯 COMPLETE THIS LINE: the total plus the loss of this photo, its squared miss
        total = ____
    # 🎯 COMPLETE THIS LINE: the average loss, the total divided by the number of photos
    return ____


print(f"average loss at w = 0 and b = 0: {average_loss(0, 0):.4f}")

In [ ]:
#@title ✅ Check task 1: three PASS lines { display-mode: "form" }
check("w = 0, b = 0: the page's first average loss", average_loss(0, 0), 64.666667)
check("w = 1, b = 2: the line misses every photo by 4 hours", average_loss(1, 2), 16.0)
check("w = 1, b = 6: the line through the three photos misses nothing", average_loss(1, 6), 0.0)

## 3. The slopes under their feet

Grasshopper $w$ stands on the number $w$, grasshopper $b$ on the number $b$. To find out which way is down, grasshopper $w$ hops sideways by $\varepsilon$, to $w + \varepsilon$, while $b$ stays where it is. The change in the average loss, divided by the hop, is the slope under its feet:

$$\frac{\partial \bar{\mathscr{L}}}{\partial w} \approx \frac{\bar{\mathscr{L}}(w + \varepsilon,\, b) - \bar{\mathscr{L}}(w,\, b)}{\varepsilon}$$

Grasshopper $b$ does the same on $b$, with $w$ back where it was:

$$\frac{\partial \bar{\mathscr{L}}}{\partial b} \approx \frac{\bar{\mathscr{L}}(w,\, b + \varepsilon) - \bar{\mathscr{L}}(w,\, b)}{\varepsilon}$$

The two slopes together are the **gradient**. The function `gradient(w, b, eps)` measures them one small step per line: the average loss where they stand, then, for each grasshopper, the average loss after its hop and the slope. At $w = 0$ and $b = 0$ it gives $-33.29$ and $-15.99$, the page's first slopes. Both are negative: the average loss goes down when either grasshopper moves to a larger number.

🎯 **Task 2.** Complete the four lines of `gradient`, two for each grasshopper:

* `loss_w`, the average loss after grasshopper $w$ hops to `w + eps`, while $b$ stays,
* `slope_w`, the change in the average loss, divided by the hop,
* `loss_b` and `slope_b`, the same for grasshopper $b$, which hops to `b + eps` while $w$ stays.

The average loss where they stand is already in `loss`. Then run the cell, and the ✅ cell below it.

<details>
<summary><b>Hint for task 2</b> (click to open)</summary>

`average_loss` gives the average loss of any line, so after the hop of grasshopper $w$ it is `average_loss(w + eps, b)`. The change is `loss_w` minus `loss`: put brackets around that subtraction, so that it comes first, and divide by `eps`. Grasshopper $b$ does the same with `b + eps`.

</details>

In [ ]:
# ============================================================
# 🎯 TASK 2: the gradient, the slope under each grasshopper
# ============================================================
def gradient(w, b, eps):
    # The slopes under the grasshoppers' feet at (w, b), each measured with a hop of eps.
    loss = average_loss(w, b)            # the average loss where they stand

    # 🎯 COMPLETE THESE TWO LINES: grasshopper w hops to w + eps while b stays
    loss_w = ____     # the average loss there
    slope_w = ____    # the change in the average loss, divided by the hop

    # 🎯 COMPLETE THESE TWO LINES: grasshopper b hops to b + eps while w stays
    loss_b = ____     # the average loss there
    slope_b = ____    # the change in the average loss, divided by the hop

    return slope_w, slope_b


first_slope_w, first_slope_b = gradient(0, 0, 0.01)
print(f"the slopes at w = 0 and b = 0: {first_slope_w:.2f} and {first_slope_b:.2f}")

In [ ]:
#@title ✅ Check task 2: three PASS lines { display-mode: "form" }
check("w = 0, b = 0, a hop of 0.01: the page's first slopes", gradient(0, 0, 0.01), (-33.286667, -15.99))
check("w = 2, b = 3, a hop of 0.1", gradient(2, 3, 0.1), (-2.2, -1.9))
check("w = 1, b = 6, a hop of 0.01: a hop forward still measures a small slope at the bottom",
      gradient(1, 6, 0.01), (0.046667, 0.01))

## 4. The two grasshoppers

One iteration goes as on the page: both grasshoppers measure the slope under their feet with `gradient`, then each jumps against its slope, by $\alpha$ times the slope:

$$w \leftarrow w - \alpha\,\frac{\partial \bar{\mathscr{L}}}{\partial w} \qquad\qquad b \leftarrow b - \alpha\,\frac{\partial \bar{\mathscr{L}}}{\partial b}$$

The loop repeats this while either slope is steeper than `tol`, $0.0005$. Below that the ground is flat enough under both: with $\alpha = 0.1$ a jump is then shorter than $0.00005$, too short to change the four decimals the page shows, and that is where the page stops too. From $w = 0$ and $b = 0$ it takes $319$ iterations, and they stop at $w = 0.9809$ and $b = 6.0330$.

🎯 **Task 3.** Complete three lines of the loop: the two slopes under their feet, from your `gradient`, then the two jumps, the new `w` and the new `b` of the two update rules above. Then run the cell, and the ✅ cell below it.

<details>
<summary><b>Hint for task 3</b> (click to open)</summary>

Call your function with where they stand and the hop, `gradient(w, b, eps)`: it gives back two numbers, and the left side of the line already splits them into `slope_w` and `slope_b`. The arrow of the slides is `=` in Python: `w = w - ...`, with $\alpha$ written `alpha` and the slope of grasshopper $w$ in `slope_w`.

</details>

The cell takes well under a second. If it is still running after a few seconds, stop it with the square button next to it, check your lines, and run the ✅ cell and part 5 to see where the grasshoppers went.

In [ ]:
w = 0.0          # where grasshopper w starts: try the start the room chose on the page
b = 0.0          # where grasshopper b starts
alpha = 0.1      # how far they jump, per unit of slope
eps = 0.01       # the test hop
tol = 0.0005     # how flat counts as flat

path = []        # where they stood at every iteration, for part 5
slope_w = 1.0    # any start steeper than tol, so that the loop starts
slope_b = 1.0
# ============================================================
# 🎯 TASK 3: the two grasshoppers, in the loop below
# ============================================================
while abs(slope_w) > tol or abs(slope_b) > tol:
    path.append((w, b))                       # where they stand
    # 🎯 COMPLETE THESE THREE LINES: the two slopes, then the two jumps
    slope_w, slope_b = ____    # the slopes under their feet, from your gradient
    w = ____                   # grasshopper w jumps against its slope
    b = ____                   # grasshopper b jumps against its slope

print(f"{len(path)} iterations, and they stand at w = {w:.4f} and b = {b:.4f}")

In [ ]:
#@title ✅ Check task 3: three PASS lines { display-mode: "form" }
# The first jump: against the two slopes measured where they started, alpha times each.
start_w, start_b = path[0]
start_slope_w, start_slope_b = gradient(start_w, start_b, eps)
first_jump = path[1] if len(path) > 1 else (w, b)
check(f"the first jump, from w = {start_w:g} and b = {start_b:g}: against each slope, alpha times it",
      first_jump, (start_w - alpha * start_slope_w, start_b - alpha * start_slope_b))

# The end: they stopped by themselves, on flat ground, and did not fly off.
last_w, last_b = path[-1]
last_slope_w, last_slope_b = gradient(last_w, last_b, eps)
flat = abs(last_slope_w) <= tol and abs(last_slope_b) <= tol
flew_off = not all(math.isfinite(v) and abs(v) < 1000 for v in (w, b, last_w, last_b))
if flew_off:
    why = (f"they flew off, to w = {w:.3g} and b = {b:.3g}: a jump the wrong way does that, "
           "and so does an alpha above 0.18")
else:
    why = "the loop stopped before the ground was flat under both of them"
check_that("they stopped by themselves, on flat ground", flat and not flew_off, why)

# The whole run, with the page's knobs.
if (path[0], alpha, eps, tol) == ((0.0, 0.0), 0.1, 0.01, 0.0005):
    check("from w = 0 and b = 0, as on the page: 319 iterations, then w = 0.9809 and b = 6.0330",
          (len(path), w, b), (319, 0.980879, 6.033002))
else:
    print("(the third check is the page's run, from w = 0 and b = 0 with alpha = 0.1, eps = 0.01")
    print(" and tol = 0.0005: you changed one of them, so it is left out)")

## 5. Did they find the line?

The line through the three photos is $w = 1$ and $b = 6$: it predicts $9$, $7$ and $8$ o'clock, the three hours exactly, so its average loss is $0$, the bottom of the valley. The cell below says first, in large letters, whether the grasshoppers found it. Then two pictures. On the left, the three photos as points, the line at the start, the line at the end, and the line through the three photos as a pale band: when they found it, the line at the end runs inside the band, and under each photo the line at the end is off by a minute or two. On the right, where the pair $(w, b)$ went, one dot per iteration, over rings of equal average loss: every point of that picture is a line, all the lines on one ring have the same average loss, and the cross marks the bottom of the valley. When they found it, the last dot sits on the cross.

Last, a table: where the grasshoppers stood at the start of some of the iterations, the average loss there and the two slopes they measured. Its first rows are the page's first iterations: at $w = 0$ and $b = 0$ the slopes are $-33.29$ and $-15.99$, and the jump takes them to $w = 3.329$ and $b = 1.599$.

In [ ]:
#@title Did they find the line? { display-mode: "form" }
BLUE, VERMILLION, INK = "#0072B2", "#D55E00", "#212121"


def banner(found, headline, detail):
    # The verdict in large letters: a blue tick when they found the line, a vermillion cross when not.
    colour, mark, ground = (BLUE, "&#10003;", "#E6F0F7") if found else (VERMILLION, "&#10007;", "#FBEAE1")
    display(HTML(
        f'<div style="border-left: 10px solid {colour}; background: {ground}; color: {INK}; '
        f'padding: 14px 20px; margin: 4px 0 14px 0; font-family: sans-serif;">'
        f'<div style="font-size: 26px; font-weight: bold; color: {colour};">{mark} {headline}</div>'
        f'<div style="font-size: 16px; margin-top: 8px;">{detail}</div></div>'))


def number(x, digits=False):
    # As on the page: four decimals, and with digits=True four significant digits
    # below 1 (the losses and the slopes); powers of ten once the grasshoppers fly off.
    if not math.isfinite(x) or abs(x) >= 1e6:
        return f"{x:12.3e}"
    if digits and 0 < abs(x) < 1:
        return f"{x:12.{min(8, 3 - math.floor(math.log10(abs(x))))}f}"
    return f"{x:12.4f}"


# The verdict. The line through the three photos is w = 1 and b = 6.
start_w, start_b = path[0]
last_w, last_b = path[-1]
flew_off = not all(math.isfinite(v) and abs(v) < 1000 for v in (w, b, last_w, last_b))
misses = []
if flew_off:
    banner(False, "They flew off",
           f"After {len(path)} iterations they stand at w = {w:.3g} and b = {b:.3g}: their jumps grew "
           "instead of shrinking. An alpha above 0.18 does that, and so does a jump the wrong way in "
           "part 4. Change it, then run part 4 and this cell again.")
else:
    last_slope_w, last_slope_b = gradient(last_w, last_b, eps)
    misses = [abs(y - (w * z2 + b)) * 60 for z2, y in photos]     # in minutes
    told = f"misses the three hours by {misses[0]:.1f}, {misses[1]:.1f} and {misses[2]:.1f} minutes"
    if abs(last_slope_w) > tol or abs(last_slope_b) > tol:
        banner(False, "They had not finished",
               f"The loop stopped after {len(path)} iterations, before the ground was flat under both "
               f"of them: the last slopes they measured are {last_slope_w:.4f} and {last_slope_b:.4f}. "
               "Run part 4 again, and let it finish.")
    elif max(misses) < 5:
        banner(True, "They found the line through the three photos",
               f"After {len(path)} iterations they stand at w = {w:.4f} and b = {b:.4f}, next to the "
               f"bottom of the valley, w = 1 and b = 6. The line at the end {told}.")
    else:
        banner(False, "They stopped beside the line through the three photos",
               f"After {len(path)} iterations they stand at w = {w:.4f} and b = {b:.4f}, and the line "
               f"at the end {told}. A long hop does that, see part 6, and so does a wrong slope in part 3.")

if not flew_off:
    fig, (left, right) = plt.subplots(1, 2, figsize=(13, 5.4))

    # Left: the photos, the line through them as a pale band, the line at the start and at the end.
    z = np.array([0.0, 4.0])
    left.plot(z, 1.0 * z + 6.0, color=BLUE, alpha=0.2, linewidth=18, solid_capstyle="butt",
              label="the line through the three photos")
    left.plot(z, start_w * z + start_b, color=BLUE, linestyle="--", linewidth=1.5,
              label="the line at the start")
    left.plot(z, w * z + b, color=BLUE, linewidth=2.5, label="the line at the end")
    left.scatter([z2 for z2, y in photos], [y for z2, y in photos], s=70, color=INK, zorder=3,
                 label="the three photos")
    for (z2, y), miss in zip(photos, misses):  # how far the line at the end is off, at each photo
        left.annotate(f"{miss:.1f} min off", (z2, y), xytext=(0, -26), textcoords="offset points",
                      ha="center", fontsize=11, fontweight="bold", color=VERMILLION if miss >= 5 else INK)
    left.set(xlim=(0, 4), ylim=(-0.5, 13), xticks=[0, 1, 2, 3, 4],
             xlabel="$z_2$, how high the sun stands", ylabel="$y$, the hour", title="The line")
    left.legend(loc="upper left")

    # Right: the pair (w, b), one dot per iteration, over rings of equal average loss.
    ws = [step[0] for step in path] + [w]
    bs = [step[1] for step in path] + [b]
    w_low, w_high = min(ws + [1.0]), max(ws + [1.0])
    b_low, b_high = min(bs + [6.0]), max(bs + [6.0])
    w_pad = 0.08 * (w_high - w_low) + 0.5
    b_pad = 0.08 * (b_high - b_low) + 0.5
    grid_w, grid_b = np.meshgrid(np.linspace(w_low - w_pad, w_high + w_pad, 400),
                                 np.linspace(b_low - b_pad, b_high + b_pad, 400))
    grid_loss = sum((y - (grid_w * z2 + grid_b)) ** 2 for z2, y in photos) / len(photos)
    rings = [m * 10.0 ** e for e in (-3, -2, -1, 0, 1, 2, 3, 4, 5, 6) for m in (1.0, 3.0)]
    right.contour(grid_w, grid_b, grid_loss, levels=rings, colors="#A8A49C", linewidths=0.8)
    right.plot(ws, bs, color=BLUE, linewidth=1, marker="o", markersize=2.5)
    right.plot(ws[0], bs[0], linestyle="none", marker="o", markersize=10, markerfacecolor="white",
               markeredgecolor=BLUE, markeredgewidth=2, label="the start")
    right.plot(ws[-1], bs[-1], linestyle="none", marker="o", markersize=10, color=BLUE,
               label="the end")
    right.plot(1.0, 6.0, linestyle="none", marker="+", markersize=22, markeredgewidth=2.5, color=INK,
               label="the bottom: w = 1, b = 6")
    right.set(xlabel="$w$", ylabel="$b$", title="Where the pair $(w, b)$ went")
    right.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

# The table: where they stood at the start of some iterations, the average loss there
# and the slopes they measured, then where they stand after the last one.
rows = [k for k in (1, 2, 3, 10, 50, 100, 200, 300, 500, 1000, 2000, 5000, 10000) if k < len(path)]
rows = rows + [len(path)]
print(f"{'iteration':>9} {'w':>12} {'b':>12} {'average loss':>14} {'slope of w':>12} {'slope of b':>12}")
for k in rows:
    path_w, path_b = path[k - 1]
    path_slope_w, path_slope_b = gradient(path_w, path_b, eps)
    print(f"{k:9d} {number(path_w)} {number(path_b)} {number(average_loss(path_w, path_b), True):>14} "
          f"{number(path_slope_w, True)} {number(path_slope_b, True)}")
print(f"after iteration {len(path)} they stand at w = {number(w).strip()} and b = {number(b).strip()}")

## 6. Other starts, steps and hops

Change the first lines of part 4, then run part 4 and part 5 again.

* The start the room chose on the page: the first rows of the table are the page's first iterations.
* $\alpha = 0.17$: fewer iterations, and the pair zigzags across the valley of the rings on its way down.
* $\alpha = 0.05$: about twice as many iterations as with $0.1$.
* $\alpha = 0.19$: they fly off, as on the page. The loop then never stops by itself: stop it with the square button next to the cell, and run part 5 to see the table.
* $\varepsilon = 0.1$: they stop near $w = 0.8$ and $b = 6.35$, beside the line through the three photos, and part 5 says so. A hop forward measures the slope about half a hop ahead of the grasshopper, so the longer the hop, the further from the bottom they stop. With $\varepsilon = 0.01$ the point they head for is ten times closer, $w = 0.98$ and $b = 6.035$, which is why the average loss grows a little again in the last rows of the table.